# Biclustering animals and features with the infinite relational model

Figure 21.20 of the book illustrates biclustering with a result from Kemp et al. (2006), *Learning systems of
concepts with an infinite relational model* (AAAI): animals and their features are clustered at the same time, so
that the animal $\times$ feature matrix falls into blocks that are mostly on or mostly off. The book reproduces the
figure from the paper; this notebook recreates it.

The data is the paper's: 50 animals $\times$ 85 binary features (Osherson et al., 1991, extended by Kemp et al.),
distributed with the *Animals with Attributes 2* dataset.

In [ ]:
import hashlib
import io
import os
import urllib.request
import zipfile

import numpy as np
from matplotlib import pyplot as plt
from scipy.special import betaln, gammaln
from sklearn.cluster import SpectralBiclustering

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## Data

`AwA2-base.zip` (32 kB) holds the animal names, the feature names and the binary matrix (the continuous ratings
thresholded at their mean).

In [ ]:
url = "https://cvml.ist.ac.at/AwA2/AwA2-base.zip"
sha256 = "3f02f8a446e81eff4f9f7d2ebb3e10b73f7c4bba3b5fd7b3442a37d38f013bcc"
data_dir = "data/awa2"
zip_path = os.path.join(data_dir, "AwA2-base.zip")
os.makedirs(data_dir, exist_ok=True)
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with open(zip_path, "rb") as fp:
    assert hashlib.sha256(fp.read()).hexdigest() == sha256, f"unexpected file: {zip_path}"

with zipfile.ZipFile(zip_path) as zf:  # read the three text files without extracting the archive
    read = lambda name: zf.read(f"Animals_with_Attributes2/{name}").decode()
    X = np.loadtxt(io.StringIO(read("predicate-matrix-binary.txt")), dtype=int)
    animals = [line.split()[1].replace("+", " ") for line in read("classes.txt").splitlines()]
    features = [line.split()[1] for line in read("predicates.txt").splitlines()]
print(f"{X.shape[0]} animals x {X.shape[1]} features, {X.mean():.0%} ones")

## The infinite relational model

For one binary matrix the IRM is a two-way clustering: animal $i$ has a cluster $z_i$, feature $j$ a cluster $w_j$,
and each block $(k, l)$ has its own probability of a 1:

$$\mathbf{z} \sim \mathrm{CRP}(\alpha), \quad \mathbf{w} \sim \mathrm{CRP}(\alpha), \quad \eta_{kl} \sim \mathrm{Beta}(\beta, \beta), \quad x_{ij} \mid \mathbf{z}, \mathbf{w}, \boldsymbol{\eta} \sim \mathrm{Ber}(\eta_{z_i w_j}).$$

The Chinese restaurant process (CRP) prior lets the number of clusters grow with the data, so it is learned rather
than fixed. Integrating out $\eta$, a block with $n^1_{kl}$ ones and $n^0_{kl}$ zeros contributes
$B(\beta + n^1_{kl}, \beta + n^0_{kl}) / B(\beta, \beta)$ to the likelihood.

**Collapsed Gibbs sampling:** remove animal $i$ from its cluster, then put it in cluster $k$ with probability
proportional to (number of other animals in $k$) $\times$ the change in likelihood, or in a new cluster with
probability proportional to $\alpha \times$ its likelihood alone; then the same for each feature (the transposed
matrix). We fix $\alpha = 1$ and $\beta = 0.5$ (Kemp et al. also sample them), run 3 chains from random
assignments to 10 clusters, and keep the state with the highest joint probability.

In [ ]:
def crp_logp(z, alpha):
    m = np.bincount(z)
    return len(m) * np.log(alpha) + gammaln(m).sum() + gammaln(alpha) - gammaln(alpha + len(z))


def log_joint(X, z, w, alpha, beta):
    Zr, Wc = np.eye(z.max() + 1)[z], np.eye(w.max() + 1)[w]
    N1 = Zr.T @ X @ Wc  # ones per block
    N0 = np.outer(Zr.sum(0), Wc.sum(0)) - N1  # zeros per block
    return crp_logp(z, alpha) + crp_logp(w, alpha) + np.sum(betaln(beta + N1, beta + N0) - betaln(beta, beta))


def relabel(z):
    return np.unique(z, return_inverse=True)[1]


def gibbs_sweep(X, z, w, alpha, beta, rng):
    """Resample the cluster of every row of X, given the column clusters w."""
    Wc = np.eye(w.max() + 1)[w]
    S, size = X @ Wc, Wc.sum(0)  # ones of each row in each column cluster, column cluster sizes
    for i in rng.permutation(len(z)):
        others = np.arange(len(z)) != i
        z_o = relabel(z[others])
        Zr = np.eye(z_o.max() + 1)[z_o]
        N1 = Zr.T @ S[others]  # [K, L]
        m = Zr.sum(0)
        N0 = m[:, None] * size - N1
        x1, x0 = S[i], size - S[i]
        logp = np.log(m) + np.sum(betaln(beta + N1 + x1, beta + N0 + x0) - betaln(beta + N1, beta + N0), axis=1)
        logp_new = np.log(alpha) + np.sum(betaln(beta + x1, beta + x0) - betaln(beta, beta))
        logp = np.append(logp, logp_new)
        p = np.exp(logp - logp.max())
        k = rng.choice(len(p), p=p / p.sum())
        z[others], z[i] = z_o, k  # k == K opens a new cluster
    return relabel(z)


def irm(X, alpha=1.0, beta=0.5, n_sweeps=200, K0=10, seed=0):
    rng = np.random.default_rng(seed)
    z, w = relabel(rng.integers(K0, size=X.shape[0])), relabel(rng.integers(K0, size=X.shape[1]))
    trace, best = [], (-np.inf, None, None)
    for _ in range(n_sweeps):
        z = gibbs_sweep(X, z, w, alpha, beta, rng)
        w = gibbs_sweep(X.T, w, z, alpha, beta, rng)
        trace.append(log_joint(X, z, w, alpha, beta))
        if trace[-1] > best[0]:
            best = (trace[-1], z.copy(), w.copy())
    return best, np.array(trace)


runs = [irm(X, seed=seed) for seed in range(3)]
traces = [trace for _, trace in runs]
lp, z, w = max((best for best, _ in runs), key=lambda b: b[0])
for seed, ((lp_s, z_s, w_s), _) in enumerate(runs):
    print(f"chain {seed}: best log joint {lp_s:.1f}, {z_s.max() + 1} animal clusters, {w_s.max() + 1} feature clusters")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for seed, (trace, color) in enumerate(zip(traces, ["#2a78d6", "#eb6834", "#1baf7a"])):
    ax.plot(np.arange(1, len(trace) + 1), trace, color=color, lw=1.5, label=f"chain {seed}")
ax.set_xlabel("Gibbs sweep")
ax.set_ylabel("log joint probability")
ax.legend()
fig.tight_layout()
pml.savefig("biclustering_irm_trace.pdf")
plt.show()

## The clusters

Clusters sorted by size. Compare with figure 21.20: O1 whales, seal, walrus, dolphin; O2 antelope, horse,
giraffe, zebra, deer; O3 monkey, gorilla, chimp; O4 hippo, elephant, rhino; O5 grizzly and polar bear; F1
flippers, strain teeth, swims, arctic, coastal, ocean, water; F2 hooves, long neck, horns; F5 meat teeth, eats
meat, hunter, fierce.

In [ ]:
z_order = np.argsort(-np.bincount(z), kind="stable")  # cluster labels, largest first
w_order = np.argsort(-np.bincount(w), kind="stable")
for n, k in enumerate(z_order, 1):
    print(f"O{n}: " + ", ".join(animals[i] for i in np.where(z == k)[0]))
print()
for n, l in enumerate(w_order, 1):
    print(f"F{n}: " + ", ".join(features[j] for j in np.where(w == l)[0]))

## The figure

The matrix with animals and features sorted by cluster (black = 1), cluster boundaries in blue, as in
figure 21.20.

In [ ]:
def plot_biclusters(ax, X, z, w, title):
    z_order = np.argsort(-np.bincount(z), kind="stable")
    w_order = np.argsort(-np.bincount(w), kind="stable")
    rows = np.concatenate([np.where(z == k)[0] for k in z_order])
    cols = np.concatenate([np.where(w == l)[0] for l in w_order])
    ax.imshow(X[np.ix_(rows, cols)], cmap="gray_r", aspect="auto", interpolation="nearest")
    for b in np.cumsum(np.bincount(z)[z_order])[:-1]:
        ax.axhline(b - 0.5, color="#2a78d6", lw=1)
    col_edges = np.cumsum(np.bincount(w)[w_order])
    for b in col_edges[:-1]:
        ax.axvline(b - 0.5, color="#2a78d6", lw=1)
    centers = (np.concatenate([[0], col_edges[:-1]]) + col_edges - 1) / 2
    ax.xaxis.tick_top()  # label the six largest feature clusters, as in figure 21.20
    ax.set_xticks(centers[:6])
    ax.set_xticklabels([f"F{n}" for n in range(1, 7)])
    ax.set_yticks(np.arange(len(rows)))
    ax.set_yticklabels([animals[i] for i in rows], fontsize=6)
    ax.set_title(title)
    return rows, cols


fig, ax = plt.subplots(figsize=(10, 7))
plot_biclusters(ax, X, z, w, f"IRM: {z.max() + 1} animal clusters x {w.max() + 1} feature clusters")
fig.tight_layout()
pml.savefig("biclustering_irm.pdf")
plt.show()

## Compared with spectral biclustering

`sklearn.cluster.SpectralBiclustering` (Kluger et al., 2003) finds the same kind of checkerboard structure, but the
numbers of row and column clusters must be given; here we give it the IRM's. Its partition scores lower under
the IRM's joint probability, and it does not choose the numbers of clusters itself.

In [ ]:
sb = SpectralBiclustering(n_clusters=(z.max() + 1, w.max() + 1), random_state=0).fit(X)
z_sb, w_sb = relabel(sb.row_labels_), relabel(sb.column_labels_)
print(f"log joint under the IRM: IRM {log_joint(X, z, w, 1.0, 0.5):.1f}, spectral {log_joint(X, z_sb, w_sb, 1.0, 0.5):.1f}")

fig, ax = plt.subplots(figsize=(10, 7))
plot_biclusters(ax, X, z_sb, w_sb, f"Spectral biclustering: {z_sb.max() + 1} x {w_sb.max() + 1} clusters (given)")
fig.tight_layout()
pml.savefig("biclustering_spectral.pdf")
plt.show()

for n, k in enumerate(np.argsort(-np.bincount(z_sb), kind="stable")[:5], 1):
    print(f"O{n}: " + ", ".join(animals[i] for i in np.where(z_sb == k)[0]))

The IRM recovers the groups shown in figure 21.20 (whales and seals, hoofed grazers, primates, the large
thick-skinned animals, the bears) together with the features that define them, and chooses the numbers of
animal and feature clusters from the data. Gibbs sampling moves one animal or feature at a time, so the chains
can settle in different local modes; Kemp et al. add split-merge moves and sample $\alpha$ and $\beta$.